## Notebook Overview

- This notebook includes an implementation of Deep Gaussian Processes following the GPyTorch documentation: https://docs.gpytorch.ai/en/stable/examples/05_Deep_Gaussian_Processes/Deep_Gaussian_Processes.html.

- The results obtained are not satisfactory and do not appear to be reliable.

In [1]:
import torch
import numpy as np
import gpytorch
import pandas as pd
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from deep_gp.deep_gaussian import DeepGPModel

from torch.utils.data import TensorDataset, DataLoader
from deep_gp.preprocessing_data import load_data, undersample_class0, apply_smote



%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:
data = load_data()
df_new = undersample_class0(data)
df_resampled = apply_smote(df_new)

X_balanced = df_resampled.drop(columns=["case_ISUP"])
y_balanced = df_resampled["case_ISUP"]
all_features = df_resampled.drop(columns=["case_ISUP"]).columns.tolist()


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42
)

# Scale features
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

# Scale target
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1,1)).ravel()


In [5]:
# convert to tensors 

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

if torch.cuda.is_available(): 
    X_train_tensor = X_train_tensor.cuda() 
    y_train_tensor = y_train_tensor.cuda() 
    X_test_tensor = X_test_tensor.cuda() 
    y_test_tensor = y_test_tensor.cuda()

In [6]:
results = []

hidden_dims = [20, 40]
num_inducing_list = [128]
learning_rates = [0.02, 0.01]

num_likelihood_samples_list = [5, 10]   

epochs = 80                               
batch_size = 64

total_models = (
    len(hidden_dims)
    * len(num_inducing_list)
    * len(learning_rates)
    * len(num_likelihood_samples_list)
)

print(f"\nTotal Deep GP models to be tested: {total_models}")



Total Deep GP models to be tested: 8


In [7]:
model_counter = 0

for hidden_dim in hidden_dims:
    for num_inducing in num_inducing_list:
        for lr in learning_rates:
            for num_likelihood_samples in num_likelihood_samples_list:

                model_counter += 1
                print(f"\n=== Model {model_counter}/{total_models} ===")
                print(f"Hidden={hidden_dim} | Inducing={num_inducing} | LR={lr} | "
                      f"Samples={num_likelihood_samples} | Epochs={epochs}")

                # DataLoaders
                train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
                test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

                # Initialize DeepGP model
                input_dim = X_train_tensor.shape[1]
                model = DeepGPModel(
                    input_dim=input_dim,
                    hidden_dim=hidden_dim,
                    num_inducing=num_inducing
                ).to(device)

                optimizer = torch.optim.Adam(model.parameters(), lr=lr)
                mll = gpytorch.mlls.DeepApproximateMLL(
                    gpytorch.mlls.VariationalELBO(
                        model.likelihood, model, num_data=X_train_tensor.shape[0]
                    )
                )

                # Training
                model.train()
                model.likelihood.train()

                for epoch in range(epochs):
                    epoch_loss = 0.0
                    for xb, yb in train_loader:
                        xb, yb = xb.to(device), yb.to(device)

                        optimizer.zero_grad()
                        with gpytorch.settings.num_likelihood_samples(num_likelihood_samples):
                            output = model(xb)
                            loss = -mll(output, yb)
                        loss.backward()
                        optimizer.step()

                        epoch_loss += loss.item()

                    if (epoch + 1) % 50 == 0:
                        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}")

                # Evaluation
                model.eval()
                model.likelihood.eval()

                with torch.no_grad():
                    pred_means_scaled, pred_vars_scaled = model.predict(
                        test_loader, num_likelihood_samples=num_likelihood_samples
                    )

                pred_means_scaled = pred_means_scaled.mean(0)
                pred_vars_scaled  = pred_vars_scaled.mean(0)

                # Back to ISUP scale
                pred_means = scaler_y.inverse_transform(
                    pred_means_scaled.cpu().numpy().reshape(-1, 1)
                ).ravel()
                y_test_orig = scaler_y.inverse_transform(
                    y_test_tensor.cpu().numpy().reshape(-1, 1)
                ).ravel()

                mse = mean_squared_error(y_test_orig, pred_means)
                r2  = r2_score(y_test_orig, pred_means)
                mean_unc = pred_vars_scaled.mean().item()

                print(f"MSE: {mse:.4f} | R²: {r2:.4f} | Mean Uncertainty: {mean_unc:.4f}")

                # Uncertainty dataframe
                vars_np = pred_vars_scaled.cpu().numpy()
                std_np  = np.sqrt(vars_np)

                df_unc = pd.DataFrame({
                    "true": y_test_orig,
                    "pred": pred_means,
                    "var": vars_np,
                    "std": std_np
                })
                df_unc["true_class"] = np.round(df_unc["true"]).astype(int)

                results.append({
                    "hidden_dim": hidden_dim,
                    "num_inducing": num_inducing,
                    "lr": lr,
                    "epochs": epochs,
                    "num_likelihood_samples": num_likelihood_samples,
                    "mse": mse,
                    "r2": r2,
                    "mean_unc": mean_unc,
                    "uncertainty": df_unc
                })



=== Model 1/8 ===
Hidden=20 | Inducing=128 | LR=0.02 | Samples=5 | Epochs=80
Epoch 50/80 - Loss: 18.5865
MSE: 3.3069 | R²: -0.0029 | Mean Uncertainty: 1.0097

=== Model 2/8 ===
Hidden=20 | Inducing=128 | LR=0.02 | Samples=10 | Epochs=80
Epoch 50/80 - Loss: 18.5547
MSE: 3.3105 | R²: -0.0040 | Mean Uncertainty: 1.0045

=== Model 3/8 ===
Hidden=20 | Inducing=128 | LR=0.01 | Samples=5 | Epochs=80
Epoch 50/80 - Loss: 18.7706
MSE: 3.3102 | R²: -0.0039 | Mean Uncertainty: 1.0510

=== Model 4/8 ===
Hidden=20 | Inducing=128 | LR=0.01 | Samples=10 | Epochs=80
Epoch 50/80 - Loss: 18.7590
MSE: 3.3109 | R²: -0.0041 | Mean Uncertainty: 1.0495

=== Model 5/8 ===
Hidden=40 | Inducing=128 | LR=0.02 | Samples=5 | Epochs=80
Epoch 50/80 - Loss: 18.5570
MSE: 3.3077 | R²: -0.0031 | Mean Uncertainty: 1.0204

=== Model 6/8 ===
Hidden=40 | Inducing=128 | LR=0.02 | Samples=10 | Epochs=80
Epoch 50/80 - Loss: 18.5806
MSE: 3.3066 | R²: -0.0028 | Mean Uncertainty: 1.0119

=== Model 7/8 ===
Hidden=40 | Inducing=128

In [8]:
df_results = pd.DataFrame(results)
df_results.head()

,hidden_dim,num_inducing,lr,epochs,num_likelihood_samples,mse,r2,mean_unc,uncertainty
0,20,128,0.02,80,5,3.306861,-0.002893,1.009658,true pred var st...
1,20,128,0.02,80,10,3.310472,-0.003988,1.004491,true pred var st...
2,20,128,0.01,80,5,3.310181,-0.003899,1.050950,true pred var std...
3,20,128,0.01,80,10,3.310870,-0.004109,1.049525,true pred var st...
4,40,128,0.02,80,5,3.307682,-0.003142,1.020431,true pred var st...


In [9]:
best = df_results.sort_values(["r2", "mean_unc"], ascending=[False, True]).iloc[0]

print("\n===== BEST DEEP GP MODEL FOUND =====")
print(f"Hidden dim:          {best['hidden_dim']}")
print(f"Inducing points:     {best['num_inducing']}")
print(f"Learning rate:       {best['lr']}")
print(f"Likelihood samples:  {best['num_likelihood_samples']}")
print(f"Epochs:              {best['epochs']}")
print(f"MSE:                 {best['mse']:.4f}")
print(f"R²:                  {best['r2']:.4f}")
print(f"Mean uncertainty:    {best['mean_unc']:.4f}")



===== BEST DEEP GP MODEL FOUND =====
Hidden dim:          40
Inducing points:     128
Learning rate:       0.02
Likelihood samples:  10
Epochs:              80
MSE:                 3.3066
R²:                  -0.0028
Mean uncertainty:    1.0119


In [10]:
df_unc_best = best["uncertainty"]

unc_by_class = df_unc_best.groupby("true_class")["std"].mean()

print("\n===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====")
for cls, std in unc_by_class.items():
    print(f"ISUP {cls}:  std = {std:.4f}")



===== MEAN UNCERTAINTY (STD) PER ISUP CLASS =====
ISUP 0:  std = 1.0059
ISUP 1:  std = 1.0059
ISUP 2:  std = 1.0059
ISUP 3:  std = 1.0059
ISUP 4:  std = 1.0059
ISUP 5:  std = 1.0059
